# Analyze an authored puzzle collection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xmaster6y/lczerolens/blob/main/docs/source/notebooks/tutorials/analyze-puzzles.ipynb)

A puzzle is a normative task with source-authored accepted continuations. This tutorial grades full-ply attempts, analyzes accepted lines exactly, and compares an evaluator's first choice without treating model preference as puzzle correctness.

In [ ]:
# Colab starts from a clean runtime; local and docs builds skip this setup.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if importlib.util.find_spec("google.colab") is not None:
    checkout = Path("/content/lczerolens")
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Xmaster6y/lczerolens.git", str(checkout)],
            check=True,
        )
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

In [ ]:
import chess

from examples.decision_analysis_tutorial import load_fixture_evaluator
from lczerolens import (
    Puzzle,
    PuzzleContinuation,
    PuzzleProvenance,
    PuzzleSolution,
    analyze_line,
)

mate_board = chess.Board("7k/8/5KQ1/8/8/8/8/8 w - - 0 1")
mate_task = Puzzle.from_board(
    mate_board,
    PuzzleSolution(
        (
            PuzzleContinuation("g6g7"),
            PuzzleContinuation("g6h6", (PuzzleContinuation("h8g8", (PuzzleContinuation("h6g7"),)),)),
        )
    ),
    provenance=PuzzleProvenance("notebook", "mate-tree"),
)
opening_task = Puzzle.from_board(
    chess.Board(),
    PuzzleSolution((PuzzleContinuation("e2e4"), PuzzleContinuation("d2d4"))),
    provenance=PuzzleProvenance("notebook", "opening-choice"),
)
puzzles = (mate_task, opening_task)

Attempts retain every ply, including authored opponent replies. A prefix can remain in progress and expose the next accepted continuation.

In [ ]:
prefix = mate_task.grade(["g6h6", "h8g8"])
solved = mate_task.grade(["g6h6", "h8g8", "h6g7"])
failed = opening_task.grade(["g1f3"])
{
    "prefix_status": prefix.status.value,
    "next_accepted": [move.uci() for move in prefix.accepted_moves],
    "full_status": solved.status.value,
    "wrong_opening_status": failed.status.value,
}

Evaluator preference is recorded alongside, not substituted for, the authored answer. Exact line analysis describes the accepted continuation's chess effects independently of either source.

In [ ]:
evaluator = load_fixture_evaluator().evaluator
evaluations = evaluator.evaluate([puzzle.position.board() for puzzle in puzzles])
rows = []
for puzzle, evaluation in zip(puzzles, evaluations):
    accepted = tuple(move.uci() for move in puzzle.accepted_moves())
    rows.append(
        {
            "puzzle": puzzle.provenance.identifier,
            "accepted": accepted,
            "model_move": evaluation.policy.best_move.uci(),
            "model_move_accepted": evaluation.policy.best_move.uci() in accepted,
        }
    )

accepted_line = analyze_line(mate_board, ["g6h6", "h8g8", "h6g7"])
assert accepted_line.moves[-1].uci() == "h6g7"
rows